In [23]:
# importing the required libraries
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import  r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

In [24]:
df=pd.read_csv('sales_revenue_prediction.csv')
df

,id,date,region,channel,product_category,customer_segment,ad_spend,price,discount_rate,market_reach,impressions,click_through_rate,competition_index,seasonality_index,campaign_duration_days,customer_lifetime_value,sales_revenue
0,1,2011-12-05 11:31:00,Nort,Search,General,Standard,9.00,0.75,0.2782,32.0,817,0.0010,3.34,1.000000,30.0,816.49,119.767811
1,2,2011-04-27 14:01:00,North,Social Media,General,Premium,3.35,3.35,0.0912,61.0,2289,0.0640,4.44,0.366025,90.0,1723.16,119.404661
2,3,2010-11-09 15:20:00,North,Affiliate,General,Budget,2.55,2.55,0.1997,461.0,14697,0.1508,3.31,0.366025,21.0,1151.74,132.009747
3,4,2010-10-03 15:24:00,North,Affiliate,Storage,Premium,2.95,2.95,0.4767,744.0,17578,0.1965,2.87,-0.366025,90.0,3585.85,154.511756
4,5,2011-10-14 09:28:00,North,Search,Lighting,Premium,15.00,1.25,0.3536,226.0,6280,0.0200,7.40,-0.366025,90.0,502.28,128.059924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23995,23996,2010-10-14 16:14:00,North,Social Media,General,Standard,3.30,1.65,0.0629,591.0,3737,0.0582,3.92,-0.366025,21.0,61.54,NaN
23996,23997,2010-12-01 11:57:00,North,Affiliate,General,Premium,3.48,0.29,0.6000,275.0,8725,0.0291,4.17,1.000000,21.0,3072.86,NaN
23997,23998,2010-03-25 16:33:00,North,Email,General,Premium,21.30,10.65,NaN,91.0,1723,0.0105,NaN,1.000000,60.0,995.19,NaN
23998,23999,2010-10-29 12:07:00,North,Social Media,Storage,Premium,45.00,1.25,0.1014,43.0,1046,0.1632,6.53,-0.366025,14.0,9856.05,NaN


In [25]:
df = df.copy()

In [26]:
# drop the columns which are not required for prediction
df.drop(['date', "id"], axis=1, inplace=True)

In [27]:
# drop duplicate rows if any
df.drop_duplicates(inplace=True)

In [28]:
# check for null values in the dataset
df.isnull().sum().sum()


np.int64(13741)

In [29]:

df = df.dropna(subset=["sales_revenue"])

df["ad_spend"] = df["ad_spend"].fillna(df["ad_spend"].median())
df["discount_rate"] = df["discount_rate"].fillna(df["discount_rate"].median())
df["market_reach"] = df["market_reach"].fillna(df["market_reach"].median())
df["competition_index"] = df["competition_index"].fillna(df["competition_index"].median())
df["click_through_rate"] = df["click_through_rate"].fillna(df["click_through_rate"].median())
df["campaign_duration_days"] = df["campaign_duration_days"].fillna(df["campaign_duration_days"].median())
df["customer_lifetime_value"] = df["customer_lifetime_value"].fillna(df["customer_lifetime_value"].median())

print(df.isnull().sum())

region                     0
channel                    0
product_category           0
customer_segment           0
ad_spend                   0
price                      0
discount_rate              0
market_reach               0
impressions                0
click_through_rate         0
competition_index          0
seasonality_index          0
campaign_duration_days     0
customer_lifetime_value    0
sales_revenue              0
dtype: int64


In [30]:
df

,region,channel,product_category,customer_segment,ad_spend,price,discount_rate,market_reach,impressions,click_through_rate,competition_index,seasonality_index,campaign_duration_days,customer_lifetime_value,sales_revenue
0,Nort,Search,General,Standard,9.00,0.75,0.2782,32.0,817,0.0010,3.34,1.000000,30.0,816.49,119.767811
1,North,Social Media,General,Premium,3.35,3.35,0.0912,61.0,2289,0.0640,4.44,0.366025,90.0,1723.16,119.404661
2,North,Affiliate,General,Budget,2.55,2.55,0.1997,461.0,14697,0.1508,3.31,0.366025,21.0,1151.74,132.009747
3,North,Affiliate,Storage,Premium,2.95,2.95,0.4767,744.0,17578,0.1965,2.87,-0.366025,90.0,3585.85,154.511756
4,North,Search,Lighting,Premium,15.00,1.25,0.3536,226.0,6280,0.0200,7.40,-0.366025,90.0,502.28,128.059924
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17995,north,Affiliate,Storage,Premium,3.30,0.55,0.2052,1115.0,41984,0.0016,8.02,-0.366025,14.0,2944.95,204.504503
17996,North,Search,Storage,Premium,5.90,2.95,0.1902,3.0,87,0.0291,5.06,-1.000000,30.0,16828.52,47.295829
17997,North,Email,General,Premium,11.90,3.75,0.0767,174.0,2605,0.0165,0.74,-1.000000,7.0,1781.44,90.457462
17998,North,Affiliate,Storage,Premium,3.75,3.75,0.0770,227.0,8761,0.0345,4.62,-0.366025,7.0,2243.36,75.691468


In [31]:
df.drop(columns=['region'], inplace=True)

In [32]:
#Label Encoding for categorical variables
le = LabelEncoder()
df['encodeproduct_category'] = le.fit_transform(df['product_category'])
var=dict(zip(df['product_category'],df['encodeproduct_category']))
print(var)
df['encodecustomer_segment'] = le.fit_transform(df['customer_segment'])
var=dict(zip(df['customer_segment'],df['encodecustomer_segment']))
print(var)
df["encodechannel"] = le.fit_transform(df["channel"])
var=dict(zip(df["channel"],df["encodechannel"]))
print(var)

{'General': 0, 'Storage': 5, 'Lighting': 2, 'Stationery': 4, 'Kitchen': 1, 'Seasonal': 3}
{'Standard': 2, 'Premium': 1, 'Budget': 0}
{'Search': 3, 'Social Media': 5, 'Affiliate': 0, 'Email': 1, 'TV': 8, 'Influencer': 2, 'Social_Media': 7, 'Socail Media': 4, 'social Media': 9, 'SocialMedia': 6, 'social media': 10}


In [33]:
#delete the original categorical columns
df.drop(['product_category', 'customer_segment', 'channel'], axis=1, inplace=True)

In [34]:
#feature selection
X = df.drop(['sales_revenue'], axis=1)
y = df['sales_revenue']

In [35]:
#splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [36]:
#scaling the features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

In [37]:
#model training using Random Forest Regressor
model = RandomForestRegressor(random_state=42)


In [38]:
para_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 8, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [0.5, 0.8, 1.0]
}

grid_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=para_grid,
    n_iter=30,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)



Fitting 5 folds for each of 30 candidates, totalling 150 fits


In [39]:
#fitting the model
best_model = grid_search.best_estimator_


In [40]:
y_pred = best_model.predict(X_test)
print("Best Parameters:", grid_search.best_params_)
print("Best CV R2:", grid_search.best_score_)
print("Test R2:", r2_score(y_test, y_pred))


Best Parameters: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 20}
Best CV R2: 0.9115081651912421
Test R2: 0.919291110553989


In [49]:
print(X.columns.tolist())

['ad_spend', 'price', 'discount_rate', 'market_reach', 'impressions', 'click_through_rate', 'competition_index', 'seasonality_index', 'campaign_duration_days', 'customer_lifetime_value', 'encodeproduct_category', 'encodecustomer_segment', 'encodechannel']


In [50]:

# Numerical inputs
ad_spend = float(input("Enter Ad Spend: "))
price = float(input("Enter Price: "))
discount_rate = float(input("Enter Discount Rate: "))
market_reach = float(input("Enter Market Reach: "))
impressions = int(input("Enter Impressions: "))
click_through_rate = float(input("Enter Click Through Rate: "))
competition_index = float(input("Enter Competition Index: "))
seasonality_index = float(input("Enter Seasonality Index: "))
campaign_duration_days = int(input("Enter Campaign Duration: "))
customer_lifetime_value = float(input("Enter Customer Lifetime Value: "))

# Product Category
print("\nProduct Category:")
print("0 = General, 1 = Kitchen, 2 = Lighting")
print("3 = Seasonal, 4 = Stationery, 5 = Storage")
product_category = int(input("Enter Category Number: "))

# Customer Segment
print("\nCustomer Segment:")
print("0 = Budget, 1 = Premium, 2 = Standard")
customer_segment = int(input("Enter Segment Number: "))

# Marketing Channel
print("\nMarketing Channel:")
print("0 = Affiliate, 1 = Email, 2 = Influencer")
print("3 = Search, 4 = Social Media, 5 = TV")
channel = int(input("Enter Channel Number: "))

# Create DataFrame
user_data = pd.DataFrame([{
    "ad_spend": ad_spend,
    "price": price,
    "discount_rate": discount_rate,
    "market_reach": market_reach,
    "impressions": impressions,
    "click_through_rate": click_through_rate,
    "competition_index": competition_index,
    "seasonality_index": seasonality_index,
    "campaign_duration_days": campaign_duration_days,
    "customer_lifetime_value": customer_lifetime_value,
    "encodeproduct_category": product_category,
    "encodecustomer_segment": customer_segment,
    "encodechannel": channel
}])

# Arrange columns in training order
user_data = user_data[X.columns]

# Predict
prediction = grid_search.best_estimator_.predict(user_data)

print("\nPredicted Sales Revenue:", prediction[0])


Product Category:
0 = General, 1 = Kitchen, 2 = Lighting
3 = Seasonal, 4 = Stationery, 5 = Storage

Customer Segment:
0 = Budget, 1 = Premium, 2 = Standard

Marketing Channel:
0 = Affiliate, 1 = Email, 2 = Influencer
3 = Search, 4 = Social Media, 5 = TV

Predicted Sales Revenue: 85.1150229566023
